[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Patrones multiagente: supervisor, traspaso y comparación.

**py501 · Aplicaciones Agénticas con LangChain** · Pythonista® · Instructor: José Luis Chiquete · Módulo 6: *LangGraph: estado, durabilidad y multiagente*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En este *notebook* se construirá un sistema multiagente con un **supervisor** que delega en agentes especialistas, se comparará contra un solo agente con todas las herramientas y se discutirá cuándo vale la pena la complejidad.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* **Ejercicio básico:** lo que todos deben dominar.
* **Reto (opcional):** para quien quiere profundizar. Se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.
* Cada ejecución llama a *Gemini* (el modelo del curso): las respuestas varían entre ejecuciones y consumen la cuota de tu `GOOGLE_API_KEY`.

### Preparación del entorno.

* La siguiente celda localizará `data/`, cargará `.env` si existe (solo en ejecución local), definirá el modelo del curso (*Gemini*) y verificará que exista la clave `GOOGLE_API_KEY`. Si falla, sigue `TUTORIAL_SECRETOS.md`.

In [ ]:
import os

from dotenv import load_dotenv

RAIZ = next(p for p in (os.curdir, os.pardir)
            if os.path.isdir(os.path.join(p, "data", "politicas")))
DATA = os.path.join(RAIZ, "data")
POLITICAS = os.path.join(DATA, "politicas")
load_dotenv(os.path.join(RAIZ, ".env"))   # solo tiene efecto en ejecución local; en Codespaces se usan los secretos

# Este curso usa Gemini. Cualquier proveedor funciona cambiando la cadena "proveedor:modelo" (ver notebook 03).
MODELO = os.environ.get("MODELO", "google_genai:gemini-3.5-flash")
EMBEDDINGS = os.environ.get("EMBEDDINGS", "google_genai:gemini-embedding-001")

if MODELO.startswith("google_genai") and not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    raise RuntimeError("Falta GOOGLE_API_KEY. Sigue TUTORIAL_SECRETOS.md: guárdala como secreto de Codespaces "
                       "con acceso a este repositorio y vuelve a abrir el Codespace.")
MODELO

* La siguiente celda importará las bibliotecas de este *notebook*.

In [ ]:
import time

from langchain.agents import create_agent
from langchain.tools import tool

* La siguiente celda definirá las herramientas de políticas.

In [ ]:
from langchain.tools import tool


@tool
def listar_politicas() -> list[str]:
    """Lista los nombres de las políticas internas disponibles."""
    return sorted(f[:-3] for f in os.listdir(POLITICAS) if f.endswith(".md"))


@tool
def leer_politica(nombre: str) -> str:
    """Devuelve el texto completo de una política interna.

    Args:
        nombre: nombre exacto devuelto por listar_politicas, sin extensión.
    """
    ruta = os.path.join(POLITICAS, f"{nombre}.md")
    if not os.path.isfile(ruta):
        return f"No existe la política '{nombre}'. Disponibles: {', '.join(listar_politicas.invoke({}))}."
    with open(ruta, encoding="utf-8") as archivo:
        return archivo.read()

* La siguiente celda conectará el servidor MCP de tickets.

In [ ]:
import sys

from langchain_mcp_adapters.client import MultiServerMCPClient

cliente_tickets = MultiServerMCPClient({
    "tickets": {
        "transport": "stdio",
        "command": sys.executable,
        "args": [os.path.join(RAIZ, "mcp_servers", "tickets_server.py")],
    }
})
herramientas_mcp = await cliente_tickets.get_tools()
[h.name for h in herramientas_mcp]

---
## 1. Patrones.

| Patrón | Cómo funciona | Ventaja | Costo |
|---|---|---|---|
| **Un solo agente** | Un agente con todas las herramientas | Simple, barato | Se confunde con muchas herramientas |
| **Supervisor** | Un agente decide a qué especialista delegar; los especialistas son herramientas del supervisor | Especialistas con contexto pequeño | Más llamadas al modelo |
| **Jerarquía** | Supervisores de supervisores | Escala a muchos dominios | Latencia y costo altos |
| **Traspaso (*handoff*)** | Un agente transfiere la conversación completa a otro | Natural para atención al cliente | Más difícil de auditar |
| **Red** | Cualquier agente puede llamar a cualquiera | Flexible | Difícil de controlar |

**Regla práctica:** empieza con un solo agente. Pasa a multiagente cuando haya **demasiadas herramientas** (más de 10 a 15), **dominios con reglas distintas** o necesidad de **modelos distintos** por tarea.

---
## 2. Supervisor con especialistas.

* La siguiente celda creará dos especialistas y un supervisor que los usa como herramientas (patrón *agente como herramienta*).

In [ ]:
especialista_politicas = create_agent(MODELO, tools=[listar_politicas, leer_politica], name="politicas",
    system_prompt="Eres experto en las políticas internas. Responde citando la política.")
especialista_tickets = create_agent(MODELO, tools=herramientas_mcp, name="tickets",
    system_prompt="Gestionas tickets de soporte. El usuario actual es 'ana.lopez'.")


@tool
async def consultar_politicas(pregunta: str) -> str:
    """Delega una duda sobre políticas internas (vacaciones, equipo, viajes) al especialista."""
    r = await especialista_politicas.ainvoke({"messages": [{"role": "user", "content": pregunta}]})
    return r["messages"][-1].text


@tool
async def gestionar_tickets(instruccion: str) -> str:
    """Delega al especialista la creación, consulta o cierre de tickets. Incluye todos los detalles."""
    r = await especialista_tickets.ainvoke({"messages": [{"role": "user", "content": instruccion}]})
    return r["messages"][-1].text


supervisor = create_agent(MODELO, tools=[consultar_politicas, gestionar_tickets], name="supervisor",
    system_prompt=("Eres el supervisor del soporte interno. No respondes directamente: delegas en "
                   "los especialistas y combinas sus respuestas."))

* La siguiente celda enviará al supervisor una solicitud que toca los dos dominios.

In [ ]:
PREGUNTA = "Se me rompió la laptop en un viaje a Monterrey. ¿Me reembolsan el taxi al taller sin factura? Y ábreme el ticket."
inicio = time.perf_counter()
r_sup = await supervisor.ainvoke({"messages": [{"role": "user", "content": PREGUNTA}]})
t_sup = time.perf_counter() - inicio
r_sup["messages"][-1].text

---
## 3. Comparación con un solo agente.

* La siguiente celda resolverá la misma solicitud con un solo agente.

In [ ]:
agente_unico = create_agent(MODELO, tools=[listar_politicas, leer_politica, *herramientas_mcp],
    system_prompt="Eres el asistente de soporte interno. El usuario actual es 'ana.lopez'.")
inicio = time.perf_counter()
r_uno = await agente_unico.ainvoke({"messages": [{"role": "user", "content": PREGUNTA}]})
t_uno = time.perf_counter() - inicio
r_uno["messages"][-1].text

* La siguiente celda comparará latencia y tokens. Los tokens de los especialistas no aparecen en los mensajes del supervisor: revísalos en la traza de *LangSmith*.

In [ ]:
def tokens(mensajes):
    return sum((m.usage_metadata or {}).get("total_tokens", 0) for m in mensajes if m.type == "ai")


{"supervisor": {"segundos": round(t_sup, 1), "tokens del supervisor": tokens(r_sup["messages"])},
 "un solo agente": {"segundos": round(t_uno, 1), "tokens": tokens(r_uno["messages"])}}

**💡 Ideas clave de la sección 3:**
* Con pocas herramientas, un solo agente suele ser más rápido y barato.
* El supervisor gana cuando los dominios tienen reglas distintas o hay muchas herramientas.
* Para comparar costos reales hay que sumar las llamadas de todos los agentes (trazas).

### Ejercicio 1 · Tercer especialista.

Agrega un especialista `viajes` con acceso solo a `leer_politica` y un *prompt* que lo limite a la política de gastos de viaje. Exponlo al supervisor como herramienta `consultar_viajes` y verifica en la traza que la pregunta del taxi se delega a él.

In [ ]:
# TU CÓDIGO AQUÍ

### Ejercicio 2 · ¿Cuándo vale la pena?

Con los números de la sección 3 y las trazas, responde: ¿en este problema vale la pena el supervisor? ¿Qué tendría que cambiar en el problema para que sí valiera la pena?

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## ✅ Cierre.

**Práctica:** terminar los ejercicios 1 y 2.

**Siguiente *notebook*:** `11_memoria_de_largo_plazo.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>